# HW2 — Multiple Linear Regression（Kaggle: Predict the Flood）

**個人作業｜CRISP-DM 六階段**。此 Notebook 以 Kaggle「Regression with a Flood Prediction Dataset」（Playground Series S4E5）的 `train.csv` 為研究資料，目標 `FloodProbability`，其餘原始預測欄位約 20 個。**請先下載真實 Kaggle 資料；未執行前不得宣稱已取得模型成果。**

資料集：https://www.kaggle.com/competitions/playground-series-s4e5/data

本作業的程式及分析由 AI 輔助草擬；最終應以實際執行輸出核對。

## 已執行的實測摘要（2026-10-14）

以使用者提供的 Kaggle 原始 train.csv（1,117,957 筆）為基礎，使用 random_state=42 抽樣 3,000 筆，再以 80/20 切分。20 個原始預測特徵、無缺失值。

- 最佳模型：Ridge(alpha=1.0)，使用全部 20 個特徵。
- 測試 MAE：0.015378；MSE：0.000385897；RMSE：0.019644；R²：0.860401；Adjusted R²：0.855579。
- OLS（全部 20 特徵）RMSE：0.019645，R²：0.860391。
- SelectKBest（14 特徵）OLS RMSE：0.033470；SFS（14 特徵）OLS RMSE：0.033681；RFE（14 特徵）OLS RMSE：0.034243。
- 範例預測：0.525323，實際值：0.525。

**重要：以上數值由另一份等價分析腳本實際計算。重新執行本 Notebook 時請使用相同原始 train.csv、相同套件設定及抽樣參數，並以當次輸出為準。**

## 1. Business Understanding（商業理解）

目標：根據環境、基礎建設等因素，建立連續數值的洪水機率預測模型，探討哪些特徵較具預測力。這是教學用資料，不能直接作為防災決策依據。

研究問題：(1) 線性迴歸在未見測試資料的誤差有多大？(2) 不同特徵選擇方法是否改善測試表現？(3) Ridge、Lasso 是否比普通最小平方法穩定？

## 2. Data Understanding（資料理解）

先檢查資料形狀、欄位、遺漏值、目標分布與相關性。**原始資料來源與 Kaggle 網址必須保留在報告中。**

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.feature_selection import SelectKBest, f_regression, RFE, SequentialFeatureSelector
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DATA_PATH = Path("data/train.csv")
TARGET = "FloodProbability"
RANDOM_STATE = 42
MAX_ROWS = 3000  # 教學運算上限；報告需揭露抽樣，設 None 可用全資料
if not DATA_PATH.exists():
    raise FileNotFoundError("請從 Kaggle 下載 train.csv，放到 HW2/data/train.csv")
df = pd.read_csv(DATA_PATH)
print("原始資料：", df.shape)
print("欄位：", df.columns.tolist())
assert TARGET in df.columns, f"找不到目標欄位 {TARGET}"
print("缺失值最多的欄位：")
display(df.isna().sum().sort_values(ascending=False).head(12).to_frame("missing"))
display(df.head())

In [ ]:
if MAX_ROWS is not None and len(df)>MAX_ROWS:
    df = df.sample(n=MAX_ROWS, random_state=RANDOM_STATE).reset_index(drop=True)
print("實際分析樣本數：", len(df))
raw_X = df.drop(columns=[TARGET, "id"], errors="ignore")
X = raw_X.select_dtypes(include="number").copy()
y = pd.to_numeric(df[TARGET], errors="coerce")
valid = y.notna()
X, y = X.loc[valid], y.loc[valid]
print("數值特徵數：", X.shape[1], "特徵名稱：", X.columns.tolist())
assert 10 <= X.shape[1] <= 39, "請確認 Kaggle 資料集與特徵數符合老師要求"
fig, ax = plt.subplots(figsize=(7,4));ax.hist(y,bins=35);ax.set(title="Target distribution",xlabel=TARGET,ylabel="Count");plt.show()
cor = X.corrwith(y).abs().sort_values(ascending=False)
display(cor.head(15).to_frame("abs correlation with target"))

## 3. Data Preparation（資料準備）

以固定亂數種子做 80% 訓練 / 20% 測試切分。遺漏值填補、標準化與特徵選擇只在**訓練集**擬合，避免測試資料洩漏。

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=RANDOM_STATE)
prep = Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler())])
Xt = prep.fit_transform(X_train)
Xv = prep.transform(X_test)
print("訓練 / 測試：",Xt.shape,Xv.shape)

## 4. Modeling（建模與特徵選擇）

比較 LinearRegression、Ridge、Lasso；特徵選擇採 SelectKBest、RFE、Sequential Forward Selection。預設 k=14；若原始特徵數不足 14，請更換資料集。特徵選擇的 fit 全部只使用訓練集。

In [ ]:
k = min(14,Xt.shape[1]-1)
assert k >= 10
selectors = {
 "All features": None,
 "SelectKBest": SelectKBest(score_func=f_regression,k=k),
 "RFE": RFE(estimator=LinearRegression(),n_features_to_select=k),
 "SFS": SequentialFeatureSelector(LinearRegression(),n_features_to_select=k,direction="forward",cv=3,n_jobs=-1)
}
models = {"OLS":LinearRegression(),"Ridge":Ridge(alpha=1.0),"Lasso":Lasso(alpha=0.001,max_iter=20000)}
results=[]; fitted={}
for selector_name, selector in selectors.items():
    if selector is None:
        Xtr,Xte=Xt,Xv; names=list(X.columns)
    else:
        selector.fit(Xt,y_train)
        Xtr,Xte=selector.transform(Xt),selector.transform(Xv)
        names=list(X.columns[selector.get_support()])
    for model_name,model in models.items():
        model.fit(Xtr,y_train)
        pred=model.predict(Xte)
        mse=mean_squared_error(y_test,pred)
        results.append({"selector":selector_name,"model":model_name,"n_features":len(names),
                        "MAE":mean_absolute_error(y_test,pred),"MSE":mse,
                        "RMSE":np.sqrt(mse),"R2":r2_score(y_test,pred),
                        "Adjusted_R2":1-(1-r2_score(y_test,pred))*(len(y_test)-1)/(len(y_test)-len(names)-1),
                        "features":", ".join(names)})
        fitted[(selector_name,model_name)] = (model,selector,names,pred)
comparison=pd.DataFrame(results).sort_values(["RMSE","R2"],ascending=[True,False])
display(comparison)
comparison.to_csv("model_comparison.csv",index=False,encoding="utf-8-sig")

## 5. Evaluation（模型評估）

使用測試集 MAE、MSE、RMSE、R²、Adjusted R² 比較。繪製真實值 vs. 預測值、殘差圖與殘差分布，並檢查線性、殘差獨立、同方差、殘差常態與多重共線性等迴歸假設。這些圖只能提供初步診斷，不能單憑視覺宣稱所有假設成立。

In [ ]:
best = comparison.iloc[0]
key=(best["selector"],best["model"])
model,selector,names,pred=fitted[key]
residual=np.asarray(y_test)-pred
fig,axs=plt.subplots(1,3,figsize=(15,4))
axs[0].scatter(y_test,pred,s=5,alpha=.3);axs[0].set(xlabel="Actual",ylabel="Predicted",title="Actual vs Predicted")
axs[1].scatter(pred,residual,s=5,alpha=.3);axs[1].axhline(0,color="black");axs[1].set(xlabel="Predicted",ylabel="Residual",title="Residual plot")
axs[2].hist(residual,bins=40);axs[2].set(title="Residual distribution",xlabel="Residual")
plt.tight_layout();plt.savefig("evaluation_plots.png",dpi=160);plt.show()
print("最佳組合：",key,"特徵：",names)
print(best[["MAE","MSE","RMSE","R2","Adjusted_R2"]])

## 5.1 95% 預測區間（老師明確要求）

採用 **OLS** 的解析式預測區間，依訓練殘差估計誤差變異與 t 分布臨界值；與 Ridge 最佳模型的評估分開呈現，避免混淆。區間的有效性依賴模型假設，並非保證每筆資料都涵蓋。

In [ ]:
# 95% prediction interval: OLS analytic variance with finite-sample t critical value.
# The interval is calculated on the held-out test set; parameters use training data only.
from scipy.stats import t as student_t
ols_pi = LinearRegression().fit(Xt, y_train)
yhat_pi = ols_pi.predict(Xv)
train_resid = np.asarray(y_train) - ols_pi.predict(Xt)
n, p = Xt.shape
sigma2 = float(np.sum(train_resid**2) / (n - p - 1))
A = np.column_stack([np.ones(n), Xt])
B = np.column_stack([np.ones(len(Xv)), Xv])
leverage = np.einsum("ij,jk,ik->i", B, np.linalg.pinv(A.T @ A), B)
critical = student_t.ppf(.975, df=n-p-1)
margin = critical * np.sqrt(sigma2 * (1 + leverage))
lo, hi = yhat_pi - margin, yhat_pi + margin
coverage = float(np.mean((np.asarray(y_test) >= lo) & (np.asarray(y_test) <= hi)))
order = np.argsort(yhat_pi)
fig, ax = plt.subplots(figsize=(11,5))
ax.plot(np.arange(len(order)), np.asarray(y_test)[order], ".", alpha=.45, ms=3, label="Actual")
ax.plot(np.arange(len(order)), yhat_pi[order], "-", lw=1.5, label="OLS prediction")
ax.fill_between(np.arange(len(order)), lo[order], hi[order], alpha=.23, label="95% prediction interval")
ax.set(xlabel="Test samples sorted by predicted value", ylabel=TARGET, title="OLS predictions with 95% prediction intervals")
ax.legend();plt.tight_layout();plt.savefig("prediction_interval_95.png", dpi=180);plt.show()
print(f"95% PI empirical coverage on test set: {coverage:.2%}")
print("Assumptions: approximate linear model, independent homoscedastic normal errors; intervals are illustrative.")

## 5.2 迴歸假設診斷

檢查線性與同方差（殘差圖、Breusch–Pagan）、常態性（Q–Q、Jarque–Bera）、獨立性（Durbin–Watson，需注意抽樣資料不具有自然時間順序）與多重共線性（VIF）。

In [ ]:
# Regression assumption diagnostics for OLS on the training split.
from scipy import stats
from statsmodels.stats.diagnostic import het_breuschpagan, acorr_breusch_godfrey
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
Xsm = sm.add_constant(Xt)
fit_sm = sm.OLS(y_train.to_numpy(), Xsm).fit()
resid_train = fit_sm.resid
bp = het_breuschpagan(resid_train, Xsm)
print("Breusch-Pagan p-value (heteroscedasticity):",bp[1])
print("Jarque-Bera p-value (residual normality):",stats.jarque_bera(resid_train).pvalue)
print("Durbin-Watson statistic (ordered observations):",sm.stats.stattools.durbin_watson(resid_train))
print("Max feature VIF:",max(variance_inflation_factor(Xt, i) for i in range(Xt.shape[1])))
fig, axes=plt.subplots(1,2,figsize=(10,4))
axes[0].scatter(fit_sm.fittedvalues, resid_train,s=4,alpha=.3);axes[0].axhline(0,color="black");axes[0].set(xlabel="Fitted",ylabel="Residual",title="Linearity / variance")
sm.qqplot(resid_train,line="45",ax=axes[1]);axes[1].set_title("Normal Q-Q")
plt.tight_layout();plt.savefig("regression_diagnostics.png",dpi=160);plt.show()
print("Caution: statistical tests may reject small departures with n=2400; independence is not established by DW alone.")

## 6. Deployment（模型部署與新資料預測）

將訓練好的預處理器、特徵選擇器與模型串接用於新樣本。此處以測試集首筆資料示範完整推論流程，**並非對未來真實事件的驗證**。

In [ ]:
new_sample=X_test.iloc[[0]]
new_transformed=prep.transform(new_sample)
if selector is not None:
    new_transformed=selector.transform(new_transformed)
new_prediction=float(model.predict(new_transformed)[0])
print("新樣本預測：",new_prediction)
print("該筆資料實際值：",float(y_test.iloc[0]))
print("注意：預測值不一定落在 0~1，因線性迴歸本身不保證機率範圍。")

## 結論與限制（執行後填寫）

- 樣本數：＿＿；原始特徵數：＿＿；訓練 / 測試：＿＿ / ＿＿。
- 最佳模型與特徵選擇：＿＿；MAE：＿＿；RMSE：＿＿；R²：＿＿。
- 重要特徵及解釋：＿＿。
- 限制：公開競賽資料未必反映實際災害風險；單次切分有隨機性；特徵相關不代表因果。

## AI 使用揭露

本 Notebook 由 ChatGPT 協助產生程式架構、CRISP-DM 段落與評估流程。請將實際 GPT 對話匯出 PDF，並附上本人以 NotebookLM 整理的至少 100 字研究摘要；**不要捏造已使用 NotebookLM 或已執行的成果**。